# inference-mode-step composite — cx15: no-grad eval pass then zero_grad before resuming training

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `inference-mode-step`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "inference-mode-step"
DD_ATOM_IDS = ["inference-mode-step", "zero-grad-set-none"]
DD_SUBTOPICS = ["PyTorch: Inference mode step", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The validation-mid-training pattern needs both atoms:

**Atom A — inference-mode-step.** Wrap the eval forward in `t.no_grad()` (or `t.inference_mode()`). This: (i) skips autograd-graph construction (cheaper, less RAM); (ii) is REQUIRED if your eval forward includes in-place leaf updates (BN running stats would error if grad tracking were on); (iii) leaves `p.grad` untouched — there is nothing to backward through.

**Atom B — zero-grad-set-none.** AFTER the eval pass, before the NEXT training step, you still need to reset `p.grad` (which is whatever it was BEFORE the eval — possibly nonzero from a prior backward you forgot to clear). Setting `p.grad = None` makes the invariant '`p.grad is None or freshly-overwritten` at the top of each train step' hold.

**Anatomy.**
```python
# ... train step finishes; p.grad may or may not have been zeroed ...
with t.no_grad():
    val_pred = model(x_val)
    val_loss = loss_fn(val_pred, y_val).item()
# Resume training: ensure clean grads BEFORE the next forward+backward.
for p in params: p.grad = None
pred = model(x_train)
loss = loss_fn(pred, y_train); loss.backward()  # safe: grad starts from None.
```

**Why both atoms together.** A subtle bug: doing the eval pass WITHOUT `t.no_grad()` builds a graph that holds onto the eval batch's activations, blowing up RAM. A second bug: skipping the post-eval `zero_grad` because 'we're under no_grad anyway' — but no_grad doesn't touch existing `p.grad`, only stops new ones from being made.

### Composite Exercise — no-grad eval pass then zero_grad before resuming training

**Atoms exercised together**: `inference-mode-step`, `zero-grad-set-none`

Implement `cx15_eval_then_resume(W, x_train, y_train, x_val, y_val, lr)`:

1. **One training step** with current `W.grad` (assume backward has ALREADY populated it). Update `W.data -= lr * W.grad` in-place.
2. **Eval pass under `t.no_grad()`**: compute `val_pred = x_val @ W`, `val_loss = ((val_pred - y_val) ** 2).mean()`. Capture `val_loss.item()` as `val_loss_v`.
3. **Zero grad with set_to_none**: set `W.grad = None`.
4. **Resume training**: compute `pred = x_train @ W`, `loss = ((pred - y_train) ** 2).mean()`, call `loss.backward()`.

Return the tuple `(val_loss_v, W.grad.clone())`.

The test verifies: (a) val_pred is computed WITHOUT a grad graph; (b) `W.grad` is None between eval and resume; (c) after resume, `W.grad` is freshly populated by the new backward (not accumulated on top of the pre-step grad).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx15_eval_then_resume(W, x_train, y_train, x_val, y_val, lr):
    """Step W with current W.grad, run no_grad eval, zero_grad, then one more backward.

    Returns (val_loss: float, fresh_W_grad: Tensor).
    """
    raise NotImplementedError

def _test_cx15():
    # Setup — populate W.grad with a known pre-step gradient.
    t.manual_seed(0)
    d, k = 4, 2
    W = t.randn(d, k, requires_grad=True)
    x_train = t.randn(10, d)
    y_train = t.randn(10, k)
    x_val = t.randn(8, d)
    y_val = t.randn(8, k)

    # Pre-step backward — populates W.grad.
    loss_pre = ((x_train @ W - y_train) ** 2).mean()
    loss_pre.backward()
    assert W.grad is not None
    pre_grad = W.grad.clone()
    W_before_step = W.data.clone()
    lr = 0.05

    val_loss_v, fresh_grad = cx15_eval_then_resume(W, x_train, y_train, x_val, y_val, lr)

    # Case A: W was updated by the pre-step grad (step happened BEFORE eval).
    expected_W_after_step = W_before_step - lr * pre_grad
    # Tolerate that the resume-backward happened AFTER the step (W is still post-step now).
    assert t.allclose(W.data, expected_W_after_step, atol=1e-6), (
        'W.data must reflect ONE step taken with the pre-eval grad'
    )

    # Case B: val_loss matches a no_grad reference computed from the post-step W.
    with t.no_grad():
        expected_val_loss = ((x_val @ W - y_val) ** 2).mean().item()
    assert abs(val_loss_v - expected_val_loss) < 1e-5, (
        f'val_loss mismatch: got {val_loss_v}, want {expected_val_loss}'
    )

    # Case C: fresh_grad came from a NEW backward (not accumulation onto pre_grad).
    # Sanity 1: shape matches.
    assert fresh_grad.shape == W.shape
    # Sanity 2: the post-eval grad does NOT equal pre_grad + something-non-zero — it equals
    # the gradient of the post-step loss alone.
    with t.no_grad():
        W_after = W.data.clone()
    ref_W = W_after.clone().requires_grad_(True)
    ref_loss = ((x_train @ ref_W - y_train) ** 2).mean()
    ref_loss.backward()
    assert t.allclose(fresh_grad, ref_W.grad, atol=1e-5), (
        'returned grad must equal the gradient of the resume forward alone (no stale accumulation)'
    )
    # Crucial inequality: fresh_grad != pre_grad + ref_W.grad — would fire if zero_grad missing.
    wrong_grad = pre_grad + ref_W.grad
    assert not t.allclose(fresh_grad, wrong_grad, atol=1e-5), (
        'fresh_grad looks like an accumulation of pre_grad + new — did you forget zero_grad?'
    )

    # Case D: W.grad is currently the fresh one (not None) because we ran backward last.
    assert W.grad is not None
    assert t.allclose(W.grad, fresh_grad), 'W.grad must be the post-resume backward result'
    _dd_passed.add('cx15')

_test_cx15()

<details><summary>Show solution — cx15</summary>

```python
def cx15_eval_then_resume(W, x_train, y_train, x_val, y_val, lr):
    # One training step using the pre-existing W.grad.
    W.data.add_(W.grad, alpha=-lr)
    # Atom A (inference-mode-step): eval forward under t.no_grad() — no graph built.
    with t.no_grad():
        val_pred = x_val @ W
        val_loss_v = ((val_pred - y_val) ** 2).mean().item()
    # Atom B (zero-grad-set-none): clear stale grad BEFORE the next backward.
    W.grad = None
    # Resume training — fresh backward populates W.grad anew.
    pred = x_train @ W
    loss = ((pred - y_train) ** 2).mean()
    loss.backward()
    return val_loss_v, W.grad.clone()
```

The load-bearing ordering: STEP → EVAL (no_grad) → ZERO_GRAD → RESUME. Move the `W.grad = None` BEFORE the eval and nothing breaks (eval doesn't touch grads under no_grad). Move it AFTER the resume backward and the next train step would skip its own zero_grad, accumulating from this step's grad. The eval pass MUST be under `t.no_grad()` — the test doesn't check that directly (we capture `.item()`), but skipping it would build a graph that holds the eval batch in memory unnecessarily.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["PyTorch: Inference mode step", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()